# Album catalog

Build a deduplicated album list from the merged review corpus (`merged_dataset.csv`).

Run `ingestion/consolidator.py` first. Outputs benchmark subsets to `datasets/`.

In [1]:
import re
import time
import unicodedata
from pathlib import Path

import pandas as pd
# import troi.http_request
from tqdm.auto import tqdm

DATA_DIR = Path("../datasets")
MERGED_PATH = DATA_DIR / "merged_dataset.csv"

## Load reviews

In [2]:
df = pd.read_csv(MERGED_PATH, low_memory=False)

print(f"Shape: {df.shape}")
print(df["dataset"].value_counts())
df.head(3)

Shape: (33079, 30)
dataset
pitchfork          22853
critique_brainz    10226
Name: count, dtype: int64


,dataset,review_id,entity_id,text,rating,album,artist,reviewer_name,reviewer_id,reviewer_type,...,review_url,is_standard_review,pub_date,body,title,score,artist_count,genres,author,cleaned_body
0,critique_brainz,60d4e2a1-53be-4ca9-9683-31db6b7cbe46,08bb6ce3-bec6-4b5e-8c6f-722e45e78fb2,I had the opportunity to attend the digital pr...,NaN,La Pluma o La Espada,Astrid Hadad,jacostamolina,70a41446-ffcf-4ebd-8892-4c3ed06f0774,Noob,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,critique_brainz,6424629d-e6b9-410a-8f7e-aa0a6f57951a,642b183a-3e59-3cb9-9187-aeeefa0d8818,Appetite for Destruction is still the better a...,4.0,Use Your Illusion I,Guns N’ Roses,smcamp1234,02fb8586-29e9-4023-9f9a-e659d222210f,Noob,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,critique_brainz,abc11944-1e57-4541-beef-23c5b3cf97b9,88727823-999e-45cb-8e43-69b26e9a670e,This is the kind of heartfelt 2010s-style tran...,3.0,AD:TRANCE,Diverse System,yorelkein,5ebf0cdb-c2a8-4170-9e78-a77b7bbd0a93,Noob,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Deduplicate albums

Normalize artist/album strings, collapse spelling variants, and count reviews per pair.

In [3]:
def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    for ch in ("\u2019", "\u2018", "\u201b", "\u2032", "`", "\u00b4"):
        s = s.replace(ch, "'")
    s = s.replace("\u2026", "...").replace("\u00b7", " ")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s)

pairs = df[["album", "artist"]].dropna().copy()
pairs["artist_norm"] = pairs["artist"].map(normalize)
pairs["album_norm"] = pairs["album"].map(normalize)
pairs.loc[pairs["artist_norm"].isin(["various", "va"]), "artist_norm"] = "various artists"

print(f"Raw unique pairs: {pairs.drop_duplicates(['artist', 'album']).shape[0]:,}")

unique_albums = (
    pairs.groupby(["artist_norm", "album_norm"], as_index=False)
    .agg(
        artist=("artist", "first"),
        album=("album", "first"),
        review_count=("album", "size"),
    )
    .sort_values(["artist", "album"])
    .reset_index(drop=True)
)

unique_albums["album_id"] = unique_albums.index
unique_albums["album"] = unique_albums["album"].str.replace("\u2019", "'", regex=False)
unique_albums["artist"] = unique_albums["artist"].str.replace("\u2019", "'", regex=False)

print(f"After dedupe: {len(unique_albums):,}")
unique_albums.head()

Raw unique pairs: 30,735
After dedupe: 30,516


,artist_norm,album_norm,artist,album,review_count,album_id
0,!!!,as if,!!!,As If,1,0
1,!!!,"jamie, my intentions are bass ep",!!!,"Jamie, My Intentions Are Bass EP",1,1
2,!!!,louden up now,!!!,Louden Up Now,1,2
3,!!!,myth takes,!!!,Myth Takes,2,3
4,!!!,shake the shudder,!!!,Shake the Shudder,1,4


## Recommendation edge list (from_to)

Map each embedding recommendation to catalog `album_id`s and save one row per edge:
`from_album_id -> to_album_id` with `rank` and `score`. Used by `lastfm-bench/lastfm_bench.ipynb` for catalog-level metrics (coverage, popularity bias, overlap).

Edges involving albums not in the current catalog are dropped — the recommendations parquet was built from a larger corpus (includes RA albums), so ~1/3 of edges have no `album_id` here.

In [4]:
RECS_PATH = DATA_DIR / "recommendations_album_level_avg_embeddings.parquet"
FROM_TO_PATH = DATA_DIR / "recommendations_from_to.csv"

recs = pd.read_parquet(RECS_PATH)

id_lookup = unique_albums.set_index(["artist_norm", "album_norm"])["album_id"]

def to_album_id(artist_col, album_col):
    keys = pd.MultiIndex.from_arrays(
        [recs[artist_col].map(normalize), recs[album_col].map(normalize)]
    )
    return pd.Series(id_lookup.reindex(keys).to_numpy(), index=recs.index)

from_to = pd.DataFrame({
    "from_album_id": to_album_id("query_artist", "query_album"),
    "to_album_id": to_album_id("rec_artist", "rec_album"),
    "rank": recs["rank"],
    "score": recs["score"],
})

n_total = len(from_to)
unmatched = from_to["from_album_id"].isna() | from_to["to_album_id"].isna()
print(f"Edges: {n_total:,}  |  unmatched (dropped): {unmatched.sum():,} ({unmatched.mean():.1%})")

from_to = from_to.dropna(subset=["from_album_id", "to_album_id"]).astype(
    {"from_album_id": int, "to_album_id": int}
)
from_to.to_csv(FROM_TO_PATH, index=False)
print(f"Saved {len(from_to):,} edges to {FROM_TO_PATH}")
from_to.head()

Edges: 447,160  |  unmatched (dropped): 162,335 (36.3%)
Saved 284,825 edges to ../datasets/recommendations_from_to.csv


,from_album_id,to_album_id,rank,score
0,0,17844,1,0.8588
1,0,17843,2,0.8443
2,0,4457,3,0.8403
3,0,10360,4,0.8352
4,0,29892,5,0.8341
